In [1]:
from copy import deepcopy
import hashlib
import json
from pathlib import Path

import mlflow
from mlflow_utils import (
    configure_mlflow,
    log_epoch_metrics,
    log_final_metrics,
    log_split_sizes,
    start_training_run,
)
import numpy as np
import polars as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.metrics import r2_score
from torch.utils.data import DataLoader, TensorDataset
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader as GeoDataLoader
from torch_geometric.nn import BatchNorm, GINEConv, global_add_pool, global_mean_pool

pl.Config.set_tbl_cols(-1)       # Wyświetl wszystkie kolumny
pl.Config.set_tbl_rows(20)       # Ustaw limit wierszy (lub -1 dla wszystkich)
pl.Config.set_fmt_str_lengths(100) # Nie skracaj długich ciągów (np. SMILES)

/home/computer/Repositories/ml_chembl/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


polars.config.Config

In [2]:
df = pl.read_parquet("processed_data/ChEMBL_processed.parquet")

In [3]:
print(df.columns)

['activity_id', 'molregno', 'canonical_smiles', 'mw_freebase', 'alogp', 'hba', 'hbd', 'psa', 'rtb', 'aromatic_rings', 'qed_weighted', 'standard_value', 'standard_units', 'standard_type', 'standard_relation', 'pchembl_value', 'target_chembl_id', 'target_name', 'confidence_score', 'pIC50']


In [4]:
radius = 2
n_bits = 2048
mfpgen = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=n_bits)

def get_scaffold_smiles(smiles: str) -> str | None:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False)

def fp_from_smiles(smiles: str) -> np.ndarray | None:
    """Zwraca fingerprint Morgana albo None dla niepoprawnego SMILES."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    try:
        return mfpgen.GetFingerprintAsNumPy(mol).astype(np.float32)
    except Exception:
        return None

class MLPBaseline(nn.Module):
    def __init__(self, input_size=2048):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1),
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        return self.network(x)

class GNNRegressor(torch.nn.Module):
    def __init__(self, node_features: int, edge_features: int, hidden_dim: int = 128, num_layers: int = 4, dropout: float = 0.15, pooling: str = 'mean'):
        super().__init__()
        if pooling not in {'mean', 'add'}:
            raise ValueError("pooling must be 'mean' or 'add'")

        self.pooling = pooling
        self.dropout = dropout
        self.node_proj = nn.Linear(node_features, hidden_dim)
        self.edge_encoder = nn.Sequential(
            nn.Linear(edge_features, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )

        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()
        for _ in range(num_layers):
            mlp = nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim),
            )
            self.convs.append(GINEConv(mlp, edge_dim=hidden_dim))
            self.norms.append(BatchNorm(hidden_dim))

        self.head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1),
        )

    def _pool(self, x, batch):
        if self.pooling == 'add':
            return global_add_pool(x, batch)
        return global_mean_pool(x, batch)

    def forward(self, data):
        x, edge_index, edge_attr, batch = data.x, data.edge_index, data.edge_attr, data.batch
        x = self.node_proj(x)
        edge_attr = self.edge_encoder(edge_attr)

        for conv, norm in zip(self.convs, self.norms):
            residual = x
            x = conv(x, edge_index, edge_attr)
            x = norm(x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = x + residual

        x = self._pool(x, batch)
        return self.head(x)


In [5]:
## Uczenie
import os
import random


def seed_everything(seed: int = 42, deterministic: bool = False):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = deterministic
    torch.backends.cudnn.benchmark = not deterministic


def get_device(prefer_cuda: bool = True):
    if prefer_cuda and torch.cuda.is_available():
        return torch.device('cuda')
    return torch.device('cpu')


def compute_num_workers(max_workers: int = 8):
    cpu_count = os.cpu_count() or 1
    return max(0, min(max_workers, cpu_count // 2))


torch.set_float32_matmul_precision('high')

device = get_device(prefer_cuda=True)
seed_everything(42, deterministic=False)
print(f'Using device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')


Using device: cuda
GPU: NVIDIA GeForce RTX 4060 Ti


In [6]:
def _forward_batch(model, batch, device, is_gnn=False):
    if is_gnn:
        batch = batch.to(device, non_blocking=True)
        target = batch.y.view(-1, 1)
        out = model(batch)
    else:
        x, y = batch
        x = x.to(device, non_blocking=True)
        target = y.to(device, non_blocking=True).view(-1, 1)
        out = model(x)
    return out, target


def train_one_epoch(model, loader, optimizer, criterion, device, is_gnn=False, clip_grad=1.0, scaler=None, use_amp=False):
    model.train()
    total_loss = 0.0

    for batch in loader:
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type=device.type, enabled=use_amp):
            out, target = _forward_batch(model, batch, device, is_gnn=is_gnn)
            loss = criterion(out, target)

        if torch.isnan(loss):
            print('Loss exploded to NaN! Stopping...')
            return float('nan')

        if scaler is not None:
            scaler.scale(loss).backward()
            if clip_grad is not None:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=clip_grad)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            if clip_grad is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=clip_grad)
            optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


def predict_arrays(model, loader, device, is_gnn=False, use_amp=False):
    model.eval()
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for batch in loader:
            with torch.amp.autocast(device_type=device.type, enabled=use_amp):
                out, target = _forward_batch(model, batch, device, is_gnn=is_gnn)
            all_preds.append(out.cpu().numpy())
            all_targets.append(target.cpu().numpy())

    preds = np.concatenate(all_preds).ravel()
    targets = np.concatenate(all_targets).ravel()
    return targets, preds


def evaluate_loss(model, loader, criterion, device, is_gnn=False, use_amp=False):
    model.eval()
    total_loss = 0.0

    with torch.no_grad():
        for batch in loader:
            with torch.amp.autocast(device_type=device.type, enabled=use_amp):
                out, target = _forward_batch(model, batch, device, is_gnn=is_gnn)
                loss = criterion(out, target)
            total_loss += loss.item()

    return total_loss / len(loader)


def evaluate_r2(model, loader, device, is_gnn=False, use_amp=False):
    targets, preds = predict_arrays(model, loader, device, is_gnn=is_gnn, use_amp=use_amp)
    if np.isnan(preds).any():
        print(f'BLAD: Predykcje modelu zawieraja NaN! (Pierwsze 5: {preds[:5]})')
    if np.isnan(targets).any():
        print('BLAD: Etykiety zawieraja NaN!')
    return r2_score(targets, preds)


In [7]:
df_temp = df.with_columns(
    pl.col("canonical_smiles").map_elements(fp_from_smiles, return_dtype=pl.Object).alias("fp")
)
df_clean = df_temp.filter(
    (pl.col("fp").is_not_null()) & 
    (pl.col("pIC50").is_not_null()) &
    (pl.col("pIC50").is_not_nan())
)
print(f"Liczba próbek po pełnym oczyszczeniu: {len(df_clean)}")
print(f"Odrzucono {len(df) - len(df_clean)} błędnych cząsteczek.")

Liczba próbek po pełnym oczyszczeniu: 15723
Odrzucono 0 błędnych cząsteczek.


In [8]:
def random_split_df(df_fp: pl.DataFrame, test_size=0.1, val_size=0.1, seed=42):
    df_shuffled = df_fp.sample(fraction=1.0, seed=seed)
    n = len(df_shuffled)
    n_test = int(test_size * n)
    n_val = int(val_size * n)
    test_df = df_shuffled[:n_test]
    val_df = df_shuffled[n_test:n_test + n_val]
    train_df = df_shuffled[n_test + n_val:]
    return train_df, val_df, test_df


def scaffold_split(df_fp: pl.DataFrame, test_size=0.1, val_size=0.1, seed=42):
    df_scaff = df_fp.with_row_index('row_id').with_columns(
        pl.col('canonical_smiles').map_elements(get_scaffold_smiles, return_dtype=pl.Utf8).alias('scaffold')
    )
    df_scaff = df_scaff.filter(pl.col('scaffold').is_not_null())

    stats = (
        df_scaff.group_by('scaffold')
        .agg(pl.len().alias('count'))
        .sort('count', descending=True)
    )

    rng = np.random.default_rng(seed)
    scaffold_rows = stats.to_dicts()
    rng.shuffle(scaffold_rows)
    scaffold_rows.sort(key=lambda r: r['count'], reverse=True)

    total = int(df_scaff.height)
    target_test = int(total * test_size)
    target_val = int(total * val_size)

    split_counts = {'train': 0, 'val': 0, 'test': 0}
    scaff_to_split = {}

    for row in scaffold_rows:
        scaf = row['scaffold']
        count = int(row['count'])
        need_test = target_test - split_counts['test']
        need_val = target_val - split_counts['val']

        if need_test > 0 and (need_test >= need_val):
            split = 'test'
        elif need_val > 0:
            split = 'val'
        else:
            split = 'train'

        scaff_to_split[scaf] = split
        split_counts[split] += count

    df_labeled = df_scaff.with_columns(
        pl.col('scaffold').replace_strict(scaff_to_split, default='train').alias('split')
    )

    return (
        df_labeled.filter(pl.col('split') == 'train'),
        df_labeled.filter(pl.col('split') == 'val'),
        df_labeled.filter(pl.col('split') == 'test'),
    )


def get_split_dfs(df_fp: pl.DataFrame, split_type='random', seed=42):
    if split_type == 'scaffold':
        return scaffold_split(df_fp, seed=seed)
    if split_type == 'random':
        return random_split_df(df_fp, seed=seed)
    raise ValueError("split_type must be 'random' or 'scaffold'")


def build_mlp_loaders(df_fp: pl.DataFrame, split_type='random', batch_size=64, seed=42):
    train_df, val_df, test_df = get_split_dfs(df_fp, split_type=split_type, seed=seed)

    X_train = np.stack(train_df['fp'].to_list())
    y_train = train_df['pIC50'].to_numpy().astype(np.float32)
    X_val = np.stack(val_df['fp'].to_list())
    y_val = val_df['pIC50'].to_numpy().astype(np.float32)
    X_test = np.stack(test_df['fp'].to_list())
    y_test = test_df['pIC50'].to_numpy().astype(np.float32)

    num_workers = compute_num_workers()
    train_loader = DataLoader(
        TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train)),
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=(device.type == 'cuda'),
        persistent_workers=(num_workers > 0),
    )
    val_loader = DataLoader(
        TensorDataset(torch.from_numpy(X_val), torch.from_numpy(y_val)),
        batch_size=batch_size,
        num_workers=num_workers,
        pin_memory=(device.type == 'cuda'),
        persistent_workers=(num_workers > 0),
    )
    test_loader = DataLoader(
        TensorDataset(torch.from_numpy(X_test), torch.from_numpy(y_test)),
        batch_size=batch_size,
        num_workers=num_workers,
        pin_memory=(device.type == 'cuda'),
        persistent_workers=(num_workers > 0),
    )
    return train_loader, val_loader, test_loader


In [9]:
ATOMIC_NUM_LIST = [1, 5, 6, 7, 8, 9, 14, 15, 16, 17, 34, 35, 53]
HYBRIDIZATION_TYPES = [
    Chem.rdchem.HybridizationType.SP,
    Chem.rdchem.HybridizationType.SP2,
    Chem.rdchem.HybridizationType.SP3,
    Chem.rdchem.HybridizationType.SP3D,
    Chem.rdchem.HybridizationType.SP3D2,
]
BOND_TYPES = [
    Chem.rdchem.BondType.SINGLE,
    Chem.rdchem.BondType.DOUBLE,
    Chem.rdchem.BondType.TRIPLE,
    Chem.rdchem.BondType.AROMATIC,
]
BOND_STEREO_TYPES = [
    Chem.rdchem.BondStereo.STEREONONE,
    Chem.rdchem.BondStereo.STEREOZ,
    Chem.rdchem.BondStereo.STEREOE,
    Chem.rdchem.BondStereo.STEREOCIS,
    Chem.rdchem.BondStereo.STEREOTRANS,
]
CHIRAL_TAGS = [
    Chem.rdchem.ChiralType.CHI_UNSPECIFIED,
    Chem.rdchem.ChiralType.CHI_TETRAHEDRAL_CW,
    Chem.rdchem.ChiralType.CHI_TETRAHEDRAL_CCW,
]

GRAPH_CACHE_DIR = Path('processed_data/graph_cache')
GRAPH_CACHE_DIR.mkdir(parents=True, exist_ok=True)
GRAPH_DATA_CACHE = GRAPH_CACHE_DIR / 'all_graphs.pt'
GRAPH_SPLIT_CACHE_TEMPLATE = 'split_{split}_seed{seed}.pt'


def one_hot_encode(value, categories):
    return [1 if value == cat else 0 for cat in categories]


def mol_to_graph(smiles: str, target: float):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    node_feats = []
    for atom in mol.GetAtoms():
        atomic_num = atom.GetAtomicNum()
        one_hot_atomic = one_hot_encode(atomic_num, ATOMIC_NUM_LIST) if atomic_num in ATOMIC_NUM_LIST else [0] * len(ATOMIC_NUM_LIST)

        hybridization = atom.GetHybridization()
        one_hot_hybrid = one_hot_encode(hybridization, HYBRIDIZATION_TYPES) if hybridization in HYBRIDIZATION_TYPES else [0] * len(HYBRIDIZATION_TYPES)

        chiral_tag = atom.GetChiralTag()
        one_hot_chiral = one_hot_encode(chiral_tag, CHIRAL_TAGS) if chiral_tag in CHIRAL_TAGS else [0] * len(CHIRAL_TAGS)

        degree = atom.GetTotalDegree() / 4.0
        formal_charge = (atom.GetFormalCharge() + 4) / 8.0
        num_hs = atom.GetTotalNumHs() / 4.0
        total_valence = atom.GetTotalValence() / 8.0
        num_radical = atom.GetNumRadicalElectrons() / 2.0
        is_aromatic = float(atom.GetIsAromatic())
        is_in_ring = float(atom.IsInRing())

        node_feats.append(
            one_hot_atomic
            + one_hot_hybrid
            + one_hot_chiral
            + [degree, formal_charge, num_hs, total_valence, num_radical, is_aromatic, is_in_ring]
        )

    x = torch.tensor(node_feats, dtype=torch.float)

    edge_index_list = []
    edge_attr_list = []
    for bond in mol.GetBonds():
        u = bond.GetBeginAtomIdx()
        v = bond.GetEndAtomIdx()

        bond_type = bond.GetBondType()
        one_hot_bond = one_hot_encode(bond_type, BOND_TYPES) if bond_type in BOND_TYPES else [0] * len(BOND_TYPES)

        stereo = bond.GetStereo()
        one_hot_stereo = one_hot_encode(stereo, BOND_STEREO_TYPES) if stereo in BOND_STEREO_TYPES else [0] * len(BOND_STEREO_TYPES)

        bond_feats = one_hot_bond + one_hot_stereo + [float(bond.GetIsConjugated()), float(bond.IsInRing())]
        edge_index_list.extend([[u, v], [v, u]])
        edge_attr_list.extend([bond_feats, bond_feats])

    edge_dim = len(BOND_TYPES) + len(BOND_STEREO_TYPES) + 2
    if edge_index_list:
        edge_index = torch.tensor(edge_index_list, dtype=torch.long).t().contiguous()
        edge_attr = torch.tensor(edge_attr_list, dtype=torch.float)
    else:
        edge_index = torch.empty((2, 0), dtype=torch.long)
        edge_attr = torch.empty((0, edge_dim), dtype=torch.float)

    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=torch.tensor([target], dtype=torch.float), smiles=smiles)


def _build_all_graphs(df_fp: pl.DataFrame):
    graphs = []
    smiles_to_idx = {}

    for row in df_fp.select(['canonical_smiles', 'pIC50']).iter_rows(named=True):
        smiles = row['canonical_smiles']
        if smiles in smiles_to_idx:
            continue
        g = mol_to_graph(smiles, float(row['pIC50']))
        if g is not None:
            smiles_to_idx[smiles] = len(graphs)
            graphs.append(g)

    return graphs, smiles_to_idx


def load_or_build_graph_cache(df_fp: pl.DataFrame):
    if GRAPH_DATA_CACHE.exists():
        try:
            cached = torch.load(GRAPH_DATA_CACHE, weights_only=False)
            return cached['graphs'], cached['smiles_to_idx']
        except Exception as exc:
            print(f'Cache load failed ({exc}); rebuilding graph cache...')

    graphs, smiles_to_idx = _build_all_graphs(df_fp)
    torch.save({'graphs': graphs, 'smiles_to_idx': smiles_to_idx}, GRAPH_DATA_CACHE)
    return graphs, smiles_to_idx


def _subset_graphs(df_part: pl.DataFrame, all_graphs, smiles_to_idx):
    graphs = []
    for smiles, target in df_part.select(['canonical_smiles', 'pIC50']).iter_rows():
        idx = smiles_to_idx.get(smiles)
        if idx is None:
            continue
        g = deepcopy(all_graphs[idx])
        g.y = torch.tensor([float(target)], dtype=torch.float)
        graphs.append(g)
    return graphs


def build_gnn_loaders(df_fp: pl.DataFrame, split_type='random', batch_size=64, seed=42):
    train_df, val_df, test_df = get_split_dfs(df_fp, split_type=split_type, seed=seed)

    all_graphs, smiles_to_idx = load_or_build_graph_cache(df_fp)

    train_graphs = _subset_graphs(train_df, all_graphs, smiles_to_idx)
    val_graphs = _subset_graphs(val_df, all_graphs, smiles_to_idx)
    test_graphs = _subset_graphs(test_df, all_graphs, smiles_to_idx)

    num_workers = compute_num_workers()
    train_loader = GeoDataLoader(
        train_graphs,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=(device.type == 'cuda'),
        persistent_workers=(num_workers > 0),
    )
    val_loader = GeoDataLoader(
        val_graphs,
        batch_size=batch_size,
        num_workers=num_workers,
        pin_memory=(device.type == 'cuda'),
        persistent_workers=(num_workers > 0),
    )
    test_loader = GeoDataLoader(
        test_graphs,
        batch_size=batch_size,
        num_workers=num_workers,
        pin_memory=(device.type == 'cuda'),
        persistent_workers=(num_workers > 0),
    )
    return train_loader, val_loader, test_loader


In [10]:
## Konfiguracja eksperymentow (uzywana przez komorki ponizej)
EPOCHS_DEFAULT = 100
LR_DEFAULT = 3e-4
BATCH_SIZE_DEFAULT = 64
SEED_DEFAULT = 42

EARLY_STOPPING_PATIENCE_DEFAULT = 12
MIN_DELTA_DEFAULT = 1e-4
WEIGHT_DECAY_DEFAULT = 1e-5
POOLING_DEFAULT = 'mean'

print(
    f'Domyslne parametry: epochs={EPOCHS_DEFAULT}, lr={LR_DEFAULT}, batch={BATCH_SIZE_DEFAULT}, '
    f'seed={SEED_DEFAULT}, patience={EARLY_STOPPING_PATIENCE_DEFAULT}, pooling={POOLING_DEFAULT}'
)


Domyslne parametry: epochs=100, lr=0.0003, batch=64, seed=42, patience=12, pooling=mean


In [11]:
fp_has_nan = any(np.isnan(fp).any() for fp in df_clean["fp"])
print(f"NaN w fingerprintach: {fp_has_nan}")

NaN w fingerprintach: False


In [12]:
# Narzedzia do pojedynczego treningu i kolekcji wynikow
results_table = []
MODEL_CACHE_DIR = Path('processed_data/model_cache')
MODEL_CACHE_DIR.mkdir(parents=True, exist_ok=True)


def _make_cache_key(config: dict) -> str:
    payload = json.dumps(config, sort_keys=True)
    return hashlib.sha256(payload.encode('utf-8')).hexdigest()[:16]


def get_model_cache_path(config: dict, namespace: str = 'default') -> Path:
    key = _make_cache_key(config)
    return MODEL_CACHE_DIR / f'{namespace}_{key}.pt'


def _upsert_result(result: dict, replace_existing: bool, model_type: str, split_type: str, seed: int, pooling: str, gnn_hidden_dim: int, gnn_num_layers: int, gnn_dropout: float, lr: float, weight_decay: float):
    if replace_existing:
        results_table[:] = [
            r
            for r in results_table
            if not (
                r['model'] == model_type
                and r['split'] == split_type
                and r.get('seed') == seed
                and r.get('pooling') == pooling
                and r.get('gnn_hidden_dim') == gnn_hidden_dim
                and r.get('gnn_num_layers') == gnn_num_layers
                and r.get('gnn_dropout') == gnn_dropout
                and r.get('lr') == lr
                and r.get('weight_decay') == weight_decay
            )
        ]
    results_table.append(result)


def train_and_score(
    model_type: str,
    split_type: str,
    epochs: int = EPOCHS_DEFAULT,
    lr: float = LR_DEFAULT,
    batch_size: int = BATCH_SIZE_DEFAULT,
    seed: int = SEED_DEFAULT,
    log_mlflow: bool = False,
    replace_existing: bool = True,
    evaluate_test: bool = False,
    early_stopping_patience: int = EARLY_STOPPING_PATIENCE_DEFAULT,
    min_delta: float = MIN_DELTA_DEFAULT,
    weight_decay: float = WEIGHT_DECAY_DEFAULT,
    pooling: str = POOLING_DEFAULT,
    gnn_hidden_dim: int = 128,
    gnn_num_layers: int = 4,
    gnn_dropout: float = 0.15,
    prefer_cuda: bool = True,
    deterministic: bool = False,
    use_amp: bool = True,
    use_model_cache: bool = True,
    force_retrain: bool = False,
    cache_namespace: str = 'default',
    mlflow_tracking_uri: str = 'sqlite:///mlflow.db',
    mlflow_experiment_name: str = 'ml_chembl_baselines',
    mlflow_artifact_root: str = 'mlruns',
):
    """Trenuje model z early stopping i wyborem najlepszego checkpointu po val_loss."""
    seed_everything(seed, deterministic=deterministic)
    device = get_device(prefer_cuda=prefer_cuda)
    amp_enabled = bool(use_amp and device.type == 'cuda')
    scaler = torch.amp.GradScaler(device='cuda', enabled=amp_enabled)

    if log_mlflow:
        configure_mlflow(
            tracking_uri=mlflow_tracking_uri,
            experiment_name=mlflow_experiment_name,
            artifact_root=mlflow_artifact_root,
        )

    if model_type == 'MLP':
        train_loader, val_loader, test_loader = build_mlp_loaders(
            df_clean, split_type=split_type, batch_size=batch_size, seed=seed
        )
        model = MLPBaseline().to(device)
        is_gnn_flag = False
    elif model_type == 'GNN':
        train_loader, val_loader, test_loader = build_gnn_loaders(
            df_clean, split_type=split_type, batch_size=batch_size, seed=seed
        )
        sample_graph = train_loader.dataset[0]
        model = GNNRegressor(
            node_features=sample_graph.num_node_features,
            edge_features=sample_graph.edge_attr.shape[1],
            hidden_dim=gnn_hidden_dim,
            num_layers=gnn_num_layers,
            dropout=gnn_dropout,
            pooling=pooling,
        ).to(device)
        is_gnn_flag = True
    else:
        raise ValueError("model_type must be 'MLP' or 'GNN'")

    cache_config = {
        'model_type': model_type,
        'split_type': split_type,
        'seed': seed,
        'epochs': epochs,
        'lr': lr,
        'batch_size': batch_size,
        'early_stopping_patience': early_stopping_patience,
        'min_delta': min_delta,
        'weight_decay': weight_decay,
        'pooling': pooling,
        'gnn_hidden_dim': gnn_hidden_dim,
        'gnn_num_layers': gnn_num_layers,
        'gnn_dropout': gnn_dropout,
    }
    model_cache_path = get_model_cache_path(cache_config, namespace=cache_namespace)

    if use_model_cache and model_cache_path.exists() and not force_retrain:
        try:
            cached = torch.load(model_cache_path, map_location=device, weights_only=False)
            model.load_state_dict(cached['model_state_dict'])

            r2_val = evaluate_r2(model, val_loader, device, is_gnn=is_gnn_flag, use_amp=amp_enabled)
            r2_test = evaluate_r2(model, test_loader, device, is_gnn=is_gnn_flag, use_amp=amp_enabled) if evaluate_test else None

            result = cached.get('result', {}).copy()
            result.update({
                'model': model_type,
                'split': split_type,
                'seed': seed,
                'r2_val': r2_val,
                'r2_test': r2_test,
                'device': str(device),
                'amp_enabled': amp_enabled,
                'from_cache': True,
                'cache_path': str(model_cache_path),
            })

            if log_mlflow:
                cached_best_val_loss = result.get('best_val_loss')
                cached_avg_train_loss = result.get('avg_train_loss')
                cached_avg_val_loss = result.get('avg_val_loss')
                with start_training_run(
                    run_name=f'{model_type}_{split_type}_seed{seed}_cache',
                    model_type=model_type,
                    split_type=split_type,
                    seed=seed,
                    extra_tags={'framework': 'pytorch', 'split_strategy': split_type, 'from_cache': 'true'},
                ):
                    mlflow.log_params({
                        'model_type': model_type,
                        'split_type': split_type,
                        'epochs': epochs,
                        'epochs_trained': result.get('epochs_trained'),
                        'best_epoch': result.get('best_epoch'),
                        'lr': lr,
                        'batch_size': batch_size,
                        'seed': seed,
                        'evaluate_test': evaluate_test,
                        'early_stopping_patience': early_stopping_patience,
                        'min_delta': min_delta,
                        'weight_decay': weight_decay,
                        'pooling': pooling,
                        'gnn_hidden_dim': gnn_hidden_dim,
                        'gnn_num_layers': gnn_num_layers,
                        'gnn_dropout': gnn_dropout,
                        'device': str(device),
                        'amp_enabled': amp_enabled,
                        'deterministic': deterministic,
                        'cache_namespace': cache_namespace,
                        'used_model_cache': use_model_cache,
                        'force_retrain': force_retrain,
                        'from_cache': True,
                    })
                    log_split_sizes(len(train_loader.dataset), len(val_loader.dataset), len(test_loader.dataset))
                    if cached_avg_train_loss is not None:
                        mlflow.log_metric('avg_train_loss', float(cached_avg_train_loss))
                    if cached_avg_val_loss is not None:
                        mlflow.log_metric('avg_val_loss', float(cached_avg_val_loss))
                    if cached_best_val_loss is not None:
                        log_final_metrics(best_val_loss=float(cached_best_val_loss), r2_val=r2_val, r2_test=r2_test)
                    else:
                        mlflow.log_metric('r2_val', float(r2_val))
                        if r2_test is not None:
                            mlflow.log_metric('r2_test', float(r2_test))

            _upsert_result(
                result=result,
                replace_existing=replace_existing,
                model_type=model_type,
                split_type=split_type,
                seed=seed,
                pooling=pooling,
                gnn_hidden_dim=gnn_hidden_dim,
                gnn_num_layers=gnn_num_layers,
                gnn_dropout=gnn_dropout,
                lr=lr,
                weight_decay=weight_decay,
            )
            print(f'Loaded cached model: {model_cache_path.name} | val R2={r2_val:.3f}')
            return result
        except Exception as exc:
            print(f'Model cache load failed ({exc}); training from scratch...')

    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=0.5,
        patience=max(2, early_stopping_patience // 3),
        min_lr=1e-6,
    )
    criterion = torch.nn.MSELoss()

    train_losses = []
    val_losses = []
    val_r2_history = []
    lr_history = []
    best_val_loss = float('inf')
    best_epoch = -1
    best_state = None
    no_improve_epochs = 0

    for epoch in range(epochs):
        epoch_train_loss = train_one_epoch(
            model,
            train_loader,
            optimizer,
            criterion,
            device,
            is_gnn=is_gnn_flag,
            clip_grad=1.0,
            scaler=scaler,
            use_amp=amp_enabled,
        )
        epoch_val_loss = evaluate_loss(model, val_loader, criterion, device, is_gnn=is_gnn_flag, use_amp=amp_enabled)
        epoch_val_r2 = evaluate_r2(model, val_loader, device, is_gnn=is_gnn_flag, use_amp=amp_enabled)
        current_lr = optimizer.param_groups[0]['lr']

        train_losses.append(epoch_train_loss)
        val_losses.append(epoch_val_loss)
        val_r2_history.append(epoch_val_r2)
        lr_history.append(current_lr)

        scheduler.step(epoch_val_loss)

        if epoch_val_loss < (best_val_loss - min_delta):
            best_val_loss = epoch_val_loss
            best_epoch = epoch + 1
            best_state = deepcopy(model.state_dict())
            no_improve_epochs = 0
        else:
            no_improve_epochs += 1

        if no_improve_epochs >= early_stopping_patience:
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    avg_train_loss = float(np.mean(train_losses))
    avg_val_loss = float(np.mean(val_losses))

    r2_val = evaluate_r2(model, val_loader, device, is_gnn=is_gnn_flag, use_amp=amp_enabled)
    r2_test = evaluate_r2(model, test_loader, device, is_gnn=is_gnn_flag, use_amp=amp_enabled) if evaluate_test else None

    if log_mlflow:
        with start_training_run(
            run_name=f'{model_type}_{split_type}_seed{seed}',
            model_type=model_type,
            split_type=split_type,
            seed=seed,
            extra_tags={'framework': 'pytorch', 'split_strategy': split_type},
        ):
            mlflow.log_params({
                'model_type': model_type,
                'split_type': split_type,
                'epochs': epochs,
                'epochs_trained': len(train_losses),
                'best_epoch': best_epoch,
                'lr': lr,
                'batch_size': batch_size,
                'seed': seed,
                'evaluate_test': evaluate_test,
                'early_stopping_patience': early_stopping_patience,
                'min_delta': min_delta,
                'weight_decay': weight_decay,
                'pooling': pooling,
                'gnn_hidden_dim': gnn_hidden_dim,
                'gnn_num_layers': gnn_num_layers,
                'gnn_dropout': gnn_dropout,
                'device': str(device),
                'amp_enabled': amp_enabled,
                'deterministic': deterministic,
                'cache_namespace': cache_namespace,
                'used_model_cache': use_model_cache,
                'force_retrain': force_retrain,
            })
            log_split_sizes(len(train_loader.dataset), len(val_loader.dataset), len(test_loader.dataset))
            mlflow.log_metric('avg_train_loss', avg_train_loss)
            mlflow.log_metric('avg_val_loss', avg_val_loss)
            log_final_metrics(best_val_loss=best_val_loss, r2_val=r2_val, r2_test=r2_test)
            epoch_history = [
                {'epoch': i, 'train_loss': tr, 'val_loss': va, 'val_r2': va_r2, 'lr': lri}
                for i, (tr, va, va_r2, lri) in enumerate(zip(train_losses, val_losses, val_r2_history, lr_history), start=1)
            ]
            log_epoch_metrics(epoch_history)
            mlflow.pytorch.log_model(model, 'model')

    result = {
        'model': model_type,
        'split': split_type,
        'seed': seed,
        'epochs': epochs,
        'epochs_trained': len(train_losses),
        'best_epoch': best_epoch,
        'lr': lr,
        'batch_size': batch_size,
        'weight_decay': weight_decay,
        'pooling': pooling,
        'gnn_hidden_dim': gnn_hidden_dim,
        'gnn_num_layers': gnn_num_layers,
        'gnn_dropout': gnn_dropout,
        'avg_train_loss': avg_train_loss,
        'avg_val_loss': avg_val_loss,
        'best_val_loss': float(best_val_loss),
        'r2_val': r2_val,
        'r2_test': r2_test,
        'device': str(device),
        'amp_enabled': amp_enabled,
        'from_cache': False,
        'cache_path': str(model_cache_path),
    }

    if use_model_cache:
        try:
            torch.save({'model_state_dict': model.state_dict(), 'result': result}, model_cache_path)
        except Exception as exc:
            print(f'Warning: could not save model cache ({exc})')

    _upsert_result(
        result=result,
        replace_existing=replace_existing,
        model_type=model_type,
        split_type=split_type,
        seed=seed,
        pooling=pooling,
        gnn_hidden_dim=gnn_hidden_dim,
        gnn_num_layers=gnn_num_layers,
        gnn_dropout=gnn_dropout,
        lr=lr,
        weight_decay=weight_decay,
    )

    if r2_test is None:
        print(
            f"{model_type} | {split_type} | device={device}: avg train loss={avg_train_loss:.4f}, avg val loss={avg_val_loss:.4f}, "
            f"best val loss={best_val_loss:.4f}, best epoch={best_epoch}, val R2={r2_val:.3f}"
        )
    else:
        print(
            f"{model_type} | {split_type} | device={device}: avg train loss={avg_train_loss:.4f}, avg val loss={avg_val_loss:.4f}, "
            f"best val loss={best_val_loss:.4f}, best epoch={best_epoch}, val R2={r2_val:.3f}, test R2={r2_test:.3f}"
        )
    return result


def tune_gnn(
    split_type: str,
    search_space: dict,
    seeds: list[int],
    epochs: int = 50,
    batch_size: int = BATCH_SIZE_DEFAULT,
    early_stopping_patience: int = EARLY_STOPPING_PATIENCE_DEFAULT,
    min_delta: float = MIN_DELTA_DEFAULT,
    log_mlflow: bool = False,
    prefer_cuda: bool = True,
):
    import itertools
    import pandas as pd

    keys = ['lr', 'weight_decay', 'pooling', 'gnn_hidden_dim', 'gnn_num_layers', 'gnn_dropout']
    combos = list(itertools.product(*(search_space[k] for k in keys)))

    tuning_rows = []
    for i, (lr, wd, pooling, hidden, layers, dropout) in enumerate(combos, start=1):
        print(
            f"[{split_type}] config {i}/{len(combos)} | lr={lr} wd={wd} pool={pooling} "
            f"hidden={hidden} layers={layers} dropout={dropout}"
        )
        per_seed = []
        for seed in seeds:
            res = train_and_score(
                model_type='GNN',
                split_type=split_type,
                epochs=epochs,
                lr=lr,
                batch_size=batch_size,
                seed=seed,
                log_mlflow=log_mlflow,
                replace_existing=False,
                evaluate_test=False,
                early_stopping_patience=early_stopping_patience,
                min_delta=min_delta,
                weight_decay=wd,
                pooling=pooling,
                gnn_hidden_dim=hidden,
                gnn_num_layers=layers,
                gnn_dropout=dropout,
                prefer_cuda=prefer_cuda,
                deterministic=False,
                use_amp=True,
                use_model_cache=True,
                force_retrain=False,
                cache_namespace='gnn_tuning',
            )
            per_seed.append(res)

        val_r2 = [r['r2_val'] for r in per_seed]
        val_loss = [r['best_val_loss'] for r in per_seed]
        tuning_rows.append({
            'model': 'GNN',
            'split': split_type,
            'lr': lr,
            'weight_decay': wd,
            'pooling': pooling,
            'gnn_hidden_dim': hidden,
            'gnn_num_layers': layers,
            'gnn_dropout': dropout,
            'seeds': ','.join(str(s) for s in seeds),
            'epochs_selection': epochs,
            'r2_val_mean': float(np.mean(val_r2)),
            'r2_val_std': float(np.std(val_r2)),
            'best_val_loss_mean': float(np.mean(val_loss)),
            'best_val_loss_std': float(np.std(val_loss)),
        })

    df_tuning = pd.DataFrame(tuning_rows).sort_values(
        ['r2_val_mean', 'best_val_loss_mean'], ascending=[False, True]
    ).reset_index(drop=True)
    return df_tuning


In [13]:
# MLP - random split (porównanie po walidacji)
train_and_score(model_type="MLP", split_type="random", log_mlflow=True, evaluate_test=False)

Loaded cached model: default_37352a75cf5257ec.pt | val R2=0.737


{'model': 'MLP',
 'split': 'random',
 'seed': 42,
 'epochs': 100,
 'epochs_trained': 73,
 'best_epoch': 61,
 'lr': 0.0003,
 'batch_size': 64,
 'weight_decay': 1e-05,
 'pooling': 'mean',
 'gnn_hidden_dim': 128,
 'gnn_num_layers': 4,
 'gnn_dropout': 0.15,
 'avg_train_loss': 0.42569581716972965,
 'avg_val_loss': 0.48538488060644236,
 'best_val_loss': 0.45433297991752625,
 'r2_val': 0.736980676651001,
 'r2_test': None,
 'device': 'cuda',
 'amp_enabled': True,
 'from_cache': True,
 'cache_path': 'processed_data/model_cache/default_37352a75cf5257ec.pt'}

In [14]:
# MLP - scaffold split (porównanie po walidacji)
train_and_score(model_type="MLP", split_type="scaffold", log_mlflow=True, evaluate_test=False)

Loaded cached model: default_1e675e4c71a847e4.pt | val R2=0.447


{'model': 'MLP',
 'split': 'scaffold',
 'seed': 42,
 'epochs': 100,
 'epochs_trained': 17,
 'best_epoch': 5,
 'lr': 0.0003,
 'batch_size': 64,
 'weight_decay': 1e-05,
 'pooling': 'mean',
 'gnn_hidden_dim': 128,
 'gnn_num_layers': 4,
 'gnn_dropout': 0.15,
 'avg_train_loss': 0.7984636863704776,
 'avg_val_loss': 0.7767915284633637,
 'best_val_loss': 0.6543190622329712,
 'r2_val': 0.4469124674797058,
 'r2_test': None,
 'device': 'cuda',
 'amp_enabled': True,
 'from_cache': True,
 'cache_path': 'processed_data/model_cache/default_1e675e4c71a847e4.pt'}

In [15]:
# GNN - uruchamiamy tylko 1 najlepsza konfiguracje na split
# Dla oceny 4.0 logujemy kazdy finalny run do MLflow.
BEST_GNN_CONFIGS = {
    'scaffold': {
        'lr': 3e-4,
        'weight_decay': 1e-5,
        'pooling': 'mean',
        'gnn_hidden_dim': 192,
        'gnn_num_layers': 4,
        'gnn_dropout': 0.10,
    },
    'random': {
        'lr': 3e-4,
        'weight_decay': 1e-5,
        'pooling': 'mean',
        'gnn_hidden_dim': 192,
        'gnn_num_layers': 4,
        'gnn_dropout': 0.10,
    },
}

gnn_selected_results = {}
for split_type, cfg in BEST_GNN_CONFIGS.items():
    gnn_selected_results[split_type] = train_and_score(
        model_type='GNN',
        split_type=split_type,
        epochs=100,
        batch_size=64,
        seed=42,
        log_mlflow=True,
        evaluate_test=False,
        early_stopping_patience=12,
        min_delta=1e-4,
        prefer_cuda=True,
        **cfg,
    )

gnn_selected_results


Loaded cached model: default_bc35dbdc16a2e430.pt | val R2=0.461
Loaded cached model: default_5d174cf5636aa4ea.pt | val R2=0.649


{'scaffold': {'model': 'GNN',
  'split': 'scaffold',
  'seed': 42,
  'epochs': 100,
  'epochs_trained': 62,
  'best_epoch': 50,
  'lr': 0.0003,
  'batch_size': 64,
  'weight_decay': 1e-05,
  'pooling': 'mean',
  'gnn_hidden_dim': 192,
  'gnn_num_layers': 4,
  'gnn_dropout': 0.1,
  'avg_train_loss': 0.8979609732744802,
  'avg_val_loss': 1.0461740172678424,
  'best_val_loss': 0.7915135335922241,
  'r2_val': 0.4610564112663269,
  'r2_test': None,
  'device': 'cuda',
  'amp_enabled': True,
  'from_cache': True,
  'cache_path': 'processed_data/model_cache/default_bc35dbdc16a2e430.pt'},
 'random': {'model': 'GNN',
  'split': 'random',
  'seed': 42,
  'epochs': 100,
  'epochs_trained': 81,
  'best_epoch': 69,
  'lr': 0.0003,
  'batch_size': 64,
  'weight_decay': 1e-05,
  'pooling': 'mean',
  'gnn_hidden_dim': 192,
  'gnn_num_layers': 4,
  'gnn_dropout': 0.1,
  'avg_train_loss': 0.8396239752977005,
  'avg_val_loss': 0.8469091184492463,
  'best_val_loss': 0.6075526428222656,
  'r2_val': 0.64925

In [16]:
# Podsumowanie wybranych konfiguracji GNN
import pandas as pd

if 'gnn_selected_results' in globals() and gnn_selected_results:
    cfg_df = pd.DataFrame([
        {'split': split, **cfg}
        for split, cfg in BEST_GNN_CONFIGS.items()
    ])
    score_df = pd.DataFrame([
        {'split': split, 'r2_val': res['r2_val'], 'best_val_loss': res['best_val_loss']}
        for split, res in gnn_selected_results.items()
    ])
    display(cfg_df.merge(score_df, on='split', how='left'))
else:
    print('Uruchom najpierw komorke z BEST_GNN_CONFIGS.')


,split,lr,weight_decay,pooling,gnn_hidden_dim,gnn_num_layers,gnn_dropout,r2_val,best_val_loss
0,scaffold,0.0003,0.00001,mean,192,4,0.1,0.461056,0.791514
1,random,0.0003,0.00001,mean,192,4,0.1,0.649254,0.607553


In [17]:
# Zestawienie wynikow w tabeli
import pandas as pd

if not results_table:
    print('Brak zebranych wynikow. Uruchom najpierw komorki treningowe.')
else:
    df_results = pd.DataFrame(results_table)
    preferred_cols = [
        'model',
        'split',
        'seed',
        'epochs',
        'epochs_trained',
        'best_epoch',
        'lr',
        'batch_size',
        'weight_decay',
        'pooling',
        'gnn_hidden_dim',
        'gnn_num_layers',
        'gnn_dropout',
        'avg_train_loss',
        'avg_val_loss',
        'best_val_loss',
        'r2_val',
        'r2_test',
        'from_cache',
        'cache_path',
    ]
    cols_to_show = [c for c in preferred_cols if c in df_results.columns]
    display(df_results[cols_to_show].sort_values(['r2_val'], ascending=False).reset_index(drop=True))


,model,split,seed,epochs,epochs_trained,best_epoch,lr,batch_size,weight_decay,pooling,gnn_hidden_dim,gnn_num_layers,gnn_dropout,avg_train_loss,avg_val_loss,best_val_loss,r2_val,r2_test,from_cache,cache_path
0,MLP,random,42,100,73,61,0.0003,64,0.00001,mean,128,4,0.15,0.425696,0.485385,0.454333,0.736981,None,True,processed_data/model_cache/default_37352a75cf5...
1,GNN,random,42,100,81,69,0.0003,64,0.00001,mean,192,4,0.10,0.839624,0.846909,0.607553,0.649254,None,True,processed_data/model_cache/default_5d174cf5636...
2,GNN,scaffold,42,100,62,50,0.0003,64,0.00001,mean,192,4,0.10,0.897961,1.046174,0.791514,0.461056,None,True,processed_data/model_cache/default_bc35dbdc16a...
3,MLP,scaffold,42,100,17,5,0.0003,64,0.00001,mean,128,4,0.15,0.798464,0.776792,0.654319,0.446912,None,True,processed_data/model_cache/default_1e675e4c71a...


In [18]:
# Finalny test uruchom raz dla najlepszego wariantu wybranego po walidacji.
# Przyklad: pobierz top konfiguracje i odpal dluzszy trening na 1 seed z testem.
# best_cfg = gnn_scaffold_tuning.iloc[0].to_dict()
# final_result = train_and_score(
#     model_type='GNN',
#     split_type='scaffold',
#     epochs=100,
#     lr=best_cfg['lr'],
#     batch_size=64,
#     seed=42,
#     weight_decay=best_cfg['weight_decay'],
#     pooling=best_cfg['pooling'],
#     gnn_hidden_dim=int(best_cfg['gnn_hidden_dim']),
#     gnn_num_layers=int(best_cfg['gnn_num_layers']),
#     gnn_dropout=best_cfg['gnn_dropout'],
#     log_mlflow=True,
#     evaluate_test=True,
#     replace_existing=False,
#     prefer_cuda=True,
#     deterministic=False,
#     use_amp=True,
# )


# Wnioski z porownania modeli

- Ranking wariantow wykonuj na podstawie **R2 walidacyjnego**, a zbior testowy uruchamiaj tylko raz dla finalisty.
- Dla GNN raportuj srednia i odchylenie standardowe po wielu seedach (minimum 3).
- Scaffold split traktuj jako glowny test uogolniania chemicznego na nowe rusztowania.
- Najpierw wybierz najlepsza konfiguracje po `gnn_scaffold_tuning`, dopiero potem uruchom finalny test.
